In [1]:
# Imports
from aiida import orm, load_profile
from aiida.engine import submit
from aiida_pythonjob import PythonJob
import ase.io
import numpy as np

# Load AiiDA profile
load_profile()

Profile<uuid='dd482d6b5e0344d1a9ba01ea32d227fe' name='default'>

In [ ]:
# Example: Create or load training structures
# In practice, you would load from an extxyz file with DFT data

# Option 1: Load from file
# train_structures = ase.io.read("training_data.xyz", ":")

# Option 2: Create dummy data for demonstration
from ase.build import bulk
from ase.units import GPa

# Create a few perturbed Si structures
train_structures = []
for i in range(5):
    atoms = bulk('Si', 'diamond', a=5.43 + i*0.01)
    atoms.rattle(stdev=0.05)
    
    # Add dummy energy and forces (replace with your DFT data)
    atoms.info['energy'] = -10.5 - i*0.1  # Total energy in eV
    atoms.arrays['forces'] = np.random.randn(len(atoms), 3) * 0.1  # Forces in eV/Å
    atoms.info['stress'] = np.random.randn(6) * 0.01 * GPa  # Stress in eV/Å³
    
    train_structures.append(atoms)

print(f"Created {len(train_structures)} training structures")

# Save training data to a file that will be used by PythonJob
# train_file = "/home/jovyan/bind_mount/work/MLIPs_PROJECT/notebooks_clustering_ft_pythonjobs/train_data.xyz"
# ase.io.write(train_file, train_structures)

In [2]:
from aiida_muon.workflows.finetuning import FineTuningWorkChain

In [3]:
# Complete workflow example
"""Complete example of MatterSim finetuning with PythonJob."""
    
# 1. Prepare training data
train_file = "/home/jovyan/bind_mount/work/MLIPs_PROJECT/notebooks_clustering_ft_pythonjobs/train_data.xyz"

# 2. Load pythonjob code
pythonjob_code = orm.load_code('python3_mattersim_p311@localhost')  # Adjust to your code label

# 3. Prepare inputs using helper function
builder = FineTuningWorkChain.get_builder_from_protocol(
        pythonjob_code = pythonjob_code,
        train_data_path=train_file,
        load_model_path="/home/jovyan/bind_mount/codes/mattersim/pretrained_models/mattersim-v1.0.0-5M.pth",
        # save_path  = './finetuned_model',
        # epochs  = 100,
        # batch_size  = 4,
        # lr  = 2e-4,
        # device = 'cpu',
        # include_forces  = True,
        # include_stresses  = False,
        # force_loss_ratio  = 1.0,
        # stress_loss_ratio  = 0.1,
        # seed  = 42,
        # pythonjob_metadata  = None,
    )



# Uncomment to run:
# node = run_finetuning_workflow()

In [4]:
# 4. Submit job
node = submit(builder)
print(f"Submitted finetuning job: PK={node.pk}")

Submitted finetuning job: PK=85569
